In [2]:
import numpy as np
import os
from typing import List, Tuple
from ipynb.fs.full.audio_parser import audio_convert, spectrogram_conversion
from ipynb.fs.full.fingerprint_maker import generate_fingerprints

Processing: Music\Lover Girl.wav...
Samples shape: (2635186,)
Sample rate: 16000
Max volume sample value: 0.0077819824
-100.0 -67.043594
Spectrogram shape: (1025, 2572)
Min/Max values in Spectrogram: -100.0 -67.043594
Jesse's function extracted 306 peaks.
Successfully generated 912 unique fingerprints!


In [ ]:
#need to finish commenting

def create_database(): #highkey redundant if not assigning independent string
    fingerprint_database = dict()
    return fingerprint_database


def add_fingerprints(database: dict, song_id: str, fingerprints: list):
    #adds songs from our library to a dictionary of fingerprints
    for (fm, fn, dt), tm in fingerprints:
        if (fm, fn, dt) not in database:
                database[(fm, fn, dt)] = []
        database[(fm, fn, dt)].append((song_id, tm))

def query_database(database: dict, query_fingerprints: list, freq_tolerance: int = 1, delta_tolerance: int = 1):
    match_counts = {}
    """
    Gets the peaks of an audio file
    Shifts it around each file in the database
    Compares the peaks across each audio file at every time window
    returns the matches
    """
    for (fm, fn, dt), tq in query_fingerprints:
        for df_fm in range(-freq_tolerance, freq_tolerance + 1):
            for df_fn in range(-freq_tolerance, freq_tolerance + 1):
                for d_dt in range(-delta_tolerance, delta_tolerance + 1):
                    fingerprint_key = (fm + df_fm, fn + df_fn, dt + d_dt)

                    if fingerprint_key in database:
                        for song_id, tm in database[fingerprint_key]:
                            time_offset = tm - tq
                            key = (song_id, time_offset)
                            match_counts[key] = match_counts.get(key, 0) + 1

    return match_counts


def get_best_match(match_counts):
    #returns song id and time offset of best match
    #need to add probability feature (if agreed on) using returned highest_count 
    #and remove time offset artifact from highest key. 
    if len(match_counts) == 0:
        return None
    
    highest_key = max(match_counts, key=match_counts.get)
    highest_count = match_counts[highest_key]
    return highest_key, highest_count


In [ ]:
""""
I commented this stuff out just in case it was still needed

song_path = os.path.join("Music", "IRIS OUT.wav") 
print(f"Processing: {song_path}...")

samples, sample_rate = audio_convert(song_path)

print("Samples shape:", samples.shape)
print("Sample rate:", sample_rate)
print("Max volume sample value:", np.max(np.abs(samples)))

log_spectro, extracted_peaks = spectrogram_conversion(samples, sample_rate)
print("Spectrogram shape:", log_spectro.shape)
print("Min/Max values in Spectrogram:", np.min(log_spectro), np.max(log_spectro))
print(f"Jesse's function extracted {len(extracted_peaks)} peaks.")

my_fingerprints = generate_fingerprints(extracted_peaks, fanout=3)
print(f"Successfully generated {len(my_fingerprints)} unique fingerprints!")
database = create_database()


#generating fingerprints for test sample
test_song_path = os.path.join("Recorded Songs", "Iris out clear.wav")
test_samples, test_sample_rate = audio_convert(test_song_path)
test_log_spectro, test_extracted_peaks = spectrogram_conversion(test_samples, test_sample_rate)
test_fingerprints = generate_fingerprints(test_extracted_peaks, fanout=3)
"""

database = create_database()

# Add fingerprints for multiple songs through loop
for song_id, path in [
    ("Iris Out", os.path.join("Music", "IRIS OUT.wav")),
    ("Billie Jean", os.path.join("Music", "Billie Jean.wav")),
    ("Lover Girl", os.path.join("Music", "Lover Girl.wav")),
    ("Black Milk", os.path.join("Music", "Black Milk.wav")),
    ("Playing God", os.path.join("Music", "Playing God.wav"))
]:
    samples, sr = audio_convert(path)
    _, peaks = spectrogram_conversion(samples, sr)
    fps = generate_fingerprints(peaks, fanout=3)
    add_fingerprints(database, song_id=song_id, fingerprints=fps)

# Query with the recorded clip (same code from earlier)
test_samples, test_sample_rate = audio_convert(os.path.join("Recorded Songs", "lover girl loud.wav"))
_, test_peaks = spectrogram_conversion(test_samples, test_sample_rate)
test_fingerprints = generate_fingerprints(test_peaks, fanout=3)

match_counts = query_database(database, test_fingerprints)
best_match = get_best_match(match_counts)
print("Best match:", best_match)

# print("Database fingerprints:", len(my_fingerprints))
print("Test fingerprints:", len(test_fingerprints))
print("Matches:", len(match_counts))

-100.0 -66.40665
-100.0 -72.34395
-100.0 -67.043594
-100.0 -66.11864
-100.0 -68.44698
-100.0 -35.768227
Best match: (('Lover Girl', np.int64(1347)), 1)
Test fingerprints: 342
Matches: 4
